# **1. Setup**



In [ ]:
!pip install transformers datasets torch scikit-learn -q

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
import numpy as np
from datasets import load_dataset, Dataset as HFDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# **2. Data Acquisition**



In [ ]:
dataset = load_dataset("UTAustin-AIHealth/MedHallu", "pqa_labeled")['train']

# Subsample for faster experiments
dataset = dataset.shuffle(seed=42).select(range(1000))

set(dataset['Category of Hallucination'])

In [ ]:
# Look at the first few examples
for i in range(4):
  print("Question: ", dataset['Question'][i])
  print("Knowledge: ", dataset['Knowledge'][i])
  print("Ground Truth: ", dataset['Ground Truth'][i])
  print("Hallucinated Answer: ", dataset['Hallucinated Answer'][i])
  print("Category of Hallucination: ", dataset['Category of Hallucination'][i])
  print()

# **3. Preprocess dataset**

In [ ]:
def preprocess(example):
    processed = []
    question = example["Question"]
    knowledge = " ".join(example["Knowledge"]) if isinstance(example["Knowledge"], list) else example["Knowledge"]
    gt_answer = example["Ground Truth"]
    hallucinated_answer = example["Hallucinated Answer"]

    if not gt_answer or not hallucinated_answer:
        return []

    premise = f"Question: {question} Context: {knowledge}"

    # Positive example (entailed)
    processed.append({"text_a": premise, "text_b": gt_answer, "label": 0})
    # Negative example (hallucinated)
    processed.append({"text_a": premise, "text_b": hallucinated_answer, "label": 1})

    return processed

processed_data = []
for ex in dataset:
    processed_data.extend(preprocess(ex))

# **4. Tokenization**

In [ ]:
checkpoint = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(checkpoint)
max_length = 256

# Tokenize manually
input_ids, attention_masks, token_type_ids, labels = [], [], [], []

for ex in processed_data:
    encoded = tokenizer(
        ex["text_a"],
        ex["text_b"],
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors="pt"
    )
    input_ids.append(encoded['input_ids'])
    attention_masks.append(encoded['attention_mask'])
    token_type_ids.append(encoded['token_type_ids'])
    labels.append(torch.tensor(ex['label']))

# Stack into tensors
input_ids = torch.cat(input_ids, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)
token_type_ids = torch.cat(token_type_ids, dim=0)
labels = torch.stack(labels)

# shuffle
perm = torch.randperm(len(labels))

input_ids = input_ids[perm]
attention_masks = attention_masks[perm]
token_type_ids = token_type_ids[perm]
labels = labels[perm]

# **5. Build final dataset**

In [ ]:
class HallucinationDataset(Dataset):
    def __init__(self, input_ids, attention_masks, token_type_ids, labels):
        self.input_ids = input_ids
        self.attention_masks = attention_masks
        self.token_type_ids = token_type_ids
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_masks[idx],
            "token_type_ids": self.token_type_ids[idx],
            "labels": self.labels[idx]
        }

# Train / validation split
split_idx = int(0.8 * len(labels))
train_dataset = HallucinationDataset(input_ids[:split_idx], attention_masks[:split_idx],
                                    token_type_ids[:split_idx], labels[:split_idx])
val_dataset = HallucinationDataset(input_ids[split_idx:], attention_masks[split_idx:],
                                token_type_ids[split_idx:], labels[split_idx:])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

# **6. BERT model & optimzer**

In [ ]:
model = BertForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
loss_fn = torch.nn.CrossEntropyLoss()

In [ ]:
for name, param in model.named_parameters():
    print(name)

# **7. Training and Evaluate**

In [ ]:
epochs = 3

print("Epoch\tTrain Loss\tVal Loss\tAcc\tF1\tPrecision\tRecall")

for epoch in range(epochs):

    # TRAIN
    model.train()
    total_train_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        batch_labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            labels=batch_labels
        )

        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # VALIDATION
    model.eval()
    total_val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch["token_type_ids"].to(device)
            batch_labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
                labels=batch_labels
            )

            loss = outputs.loss
            logits = outputs.logits

            total_val_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            all_labels.extend(batch_labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    avg_val_loss = total_val_loss / len(val_loader)

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='binary'
    )

    acc = np.mean(np.array(all_labels) == np.array(all_preds))

    print(f"{epoch+1}\t{avg_train_loss:.4f}\t\t{avg_val_loss:.4f}\t\t{acc:.4f}\t{f1:.4f}\t{precision:.4f}\t{recall:.4f}")

    cm = confusion_matrix(all_labels, all_preds)
    print("Confusion Matrix:\n", cm)